> ℹ️ This notebook is adapted from Unsloth (LGPL-3.0). Modifications were made.
---
---



To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it


## Setup

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install hf_transfer
!pip install trackio==0.12.0

In [6]:
import os
import torch
from typing import Literal
from dataclasses import dataclass, asdict, field

@dataclass
class EnvConfig:
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    is_kaggle: bool = "KAGGLE_" in "".join(os.environ.keys())
    requires_keys: bool = True
    enable_trackio: bool = True
    test_size = 0.1

@dataclass
class DataConfig:
    gold_dataset_repo_id: str = "Francesco-A/clinical-policy-gold-distill-cot"

@dataclass
class ModelConfig:
    checkpoint: str = "unsloth/Qwen3-14B"
    fine_tuned_model_repo_id: str = "Francesco-A/Qwen3-14B-Clinical-Reasoning-LoRA"
    push_to_hub: bool = True
    save_local: bool = False
    project_name: str = "sft-tests"
    trackio_space_id: str = "trackio"
    training_mode: Literal["epoch", "steps", "smoke_test"] = "epoch" # "epoch", "steps", "smoke_test"
    resume_training: bool = False
    num_train_epochs: int = 3
    batch_size: int = 1
    gradient_acc: int = 4


@dataclass
# NOTE: Use default_factory for mutable default values to prevent unintended shared state.
class CFG:
    env: EnvConfig = field(default_factory=EnvConfig)
    data: DataConfig = field(default_factory=DataConfig)
    model: ModelConfig = field(default_factory=ModelConfig)

    def __post_init__(self):
        """Runs automatically after the class is initialized."""
        os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
        os.environ["HF_HUB_DISABLE_XET"] = "1"
        if self.env.enable_trackio:
            os.environ["TRACKIO_SPACE_ID"] = self.model.trackio_space_id

        # 1. Load keys (if necessary)
        if self.env.requires_keys:
            if self.env.is_kaggle:
                try:
                    from kaggle_secrets import UserSecretsClient
                    os.environ["HUGGINGFACE_READ_TOKEN"] = UserSecretsClient().get_secret("HUGGINGFACE_READ_TOKEN")
                    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HUGGINGFACE_WRITE_TOKEN")
                    print("✅ All Kaggle Secrets loaded.")
                except Exception as e:
                    print(f"ℹ️ Kaggle secrets not found or an error occurred: {e}")
            else:
                try:
                    from google.colab import userdata
                    os.environ["HUGGINGFACE_READ_TOKEN"] = userdata.get("HUGGINGFACE_READ_TOKEN")
                    os.environ["HF_TOKEN"] = userdata.get("HF_WRITE_TOKEN")
                    print("✅ All Colab Userdata Secrets loaded.")
                except Exception as e:
                    print(f"ℹ️ Colab secrets not found or an error occurred: {e}")

    def to_dict(self):
        return asdict(self)

In [8]:
from pprint import pprint

# Initialize config
config = CFG()
if config.model.training_mode == 'smoke_test':
    print("ℹ️Running in smoke test mode!")

print(f"Using device: {config.env.device}")
pprint(config.to_dict())

✅ All Kaggle Secrets loaded.
Using device: cuda
{'data': {'gold_dataset_repo_id': 'Francesco-A/clinical-policy-gold-distill-cot'},
 'env': {'device': 'cuda',
         'enable_trackio': True,
         'is_kaggle': True,
         'requires_keys': True,
         'seed': 42},
 'model': {'batch_size': 1,
           'checkpoint': 'unsloth/Qwen3-14B',
           'fine_tuned_model_repo_id': 'Francesco-A/Qwen3-14B-Clinical-Reasoning-LoRA',
           'gradient_acc': 4,
           'num_train_epochs': 3,
           'project_name': 'sft-tests',
           'push_to_hub': True,
           'resume_training': False,
           'save_local': False,
           'trackio_space_id': 'trackio',
           'training_mode': 'epoch'}}


## src

### src/utils

In [9]:
def generate_conversation(examples, user_key, assistant_key):
    conversations = []
    for u, a in zip(examples[user_key], examples[assistant_key]):
        conversations.append([
            {"role": "user", "content": u},
            {"role": "assistant", "content": a},
        ])
    return {"conversations": conversations}

def simplify_prompt(text):
    title_start = text.find("TITLE:")
    summary_start = text.find("SUMMARY:")

    title = text[title_start:summary_start].strip()
    summary = text[summary_start:].strip()

    return (
        "Summarize the following clinical trial report for policy makers:\n\n"
        f"{title}\n{summary}"
    )

## Unsloth

In [10]:
from huggingface_hub import snapshot_download

print("Starting high-speed download...")
snapshot_download(repo_id=config.model.checkpoint)
print("✅ Model cached successfully!")

Starting high-speed download...


.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

... (11 outputs left out)

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

✅ Model cached successfully!


In [11]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-1.7B-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",

    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = config.model.checkpoint,
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "YOUR_HF_TOKEN",      # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-04-20 15:39:01.789445: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776699542.235376      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776699542.384201      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776699543.809540      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776699543.809573      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776699543.809575      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.59G [00:00<?, ?B/s]

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-14b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = config.env.seed,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2026.4.6 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


<a name="Data"></a>
## Data Prep
Qwen3 has both reasoning and a non reasoning mode. So, we should use 2 datasets:

1. We use the [Open Math Reasoning](https://huggingface.co/datasets/unsloth/OpenMathReasoning-mini) dataset which was used to win the [AIMO](https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-2/leaderboard) (AI Mathematical Olympiad - Progress Prize 2) challenge! We sample 10% of verifiable reasoning traces that used DeepSeek R1, and which got > 95% accuracy.

2. We also leverage [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. But we need to convert it to HuggingFace's normal multiturn format as well.

In [13]:
from datasets import load_dataset
clinical_trial_dataset = load_dataset(config.data.gold_dataset_repo_id, split = "train")
reasoning_dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
non_reasoning_dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/903k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/191 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/603 [00:00<?, ?B/s]

cot-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

Generating cot split:   0%|          | 0/19252 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Let's see the structure of all datasets:

In [14]:
clinical_trial_dataset

Dataset({
    features: ['nctId', 'keyword', 'prompt', 'document', 'target_audience', 'generations', 'model_names', 'distilabel_metadata', 'ratings', 'rationales', 'judge_model', '__index_level_0__'],
    num_rows: 191
})

In [15]:
reasoning_dataset

Dataset({
    features: ['expected_answer', 'problem_type', 'problem_source', 'generation_model', 'pass_rate_72b_tir', 'problem', 'generated_solution', 'inference_mode'],
    num_rows: 19252
})

In [16]:
non_reasoning_dataset

Dataset({
    features: ['conversations', 'source', 'score'],
    num_rows: 100000
})

Clean dataset (if necessary): here we need to clean the overly engineered `clinical_trial_dataset` prompt into something more likely to be used during actual inference

In [17]:
clinical_trial_dataset = clinical_trial_dataset.map(
    lambda examples: {"prompt": simplify_prompt(examples["prompt"])}
    )

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

We now convert the reasoning dataset into conversational format:

In [18]:
clinical_summaries = tokenizer.apply_chat_template(
    list(clinical_trial_dataset.map(
        lambda examples: generate_conversation(
            examples,
            user_key="prompt",
            assistant_key="generations"
            ), batched = True)["conversations"]),
    tokenize = False,
)

reasoning_conversations = tokenizer.apply_chat_template(
    list(reasoning_dataset.map(
        lambda examples: generate_conversation(
            examples,
            user_key="problem",
            assistant_key="generated_solution"
            ), batched = True)["conversations"]),
    tokenize = False,
)

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

Map:   0%|          | 0/19252 [00:00<?, ? examples/s]

Let's see the first transformed row:

In [19]:
clinical_summaries[0]

'<|im_start|>user\nSummarize the following clinical trial report for policy makers:\n\nTITLE: Danish Cardiogenic Shock Trial\nSUMMARY: Cardiogenic shock a serious complication of a heart attack (myocardial infarction). Despite rapid invasive treatment, circulatory support using positive inotropes and mechanical support with intra-aortic balloon counterpulsation (IABP), and evaluation of several new treatments during the last decade, the mortality in patients with cardiogenic shock still exceeds 50%. An alternative to current management is restoration of the volume of blood pumped by the heart (cardiac output) using a ventricular assist device. In the acute setting this is difficult but can be done using the Impella device which is a catheter-based, axial flow pump that pumps blood directly from the left ventricle into the circulation thereby restoring blood flow to the failing organs. In 2012 a more powerful Impella has been introduced that is able to deliver 3.5l/min (approximately 75

In [20]:
reasoning_conversations[0]

"<|im_start|>user\nGiven $\\sqrt{x^2+165}-\\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.\n\nFirst, let me write down the equation again to make sure I have it right:\n\n√(x² + 165) - √(x² - 52) = 7.\n\nOkay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:\n\n√(x² + 165) = 7 + √(x² - 52).\n\nNow, if I square both sides, maybe I can get rid of the square roots. Let's do that:\n\n(√(x² + 165))² = (7 + √(x² - 52))².\n\nSimplifying the left side:\n\nx² + 165 = 49 + 14√(x² - 52) + (√(x² - 52))².\n\nThe right side is expanded using the formula (a + b)² = a² + 2ab + b². So the right side becomes 7² + 2*7*√(x² - 52) + (√(x² 

Next we take the non reasoning dataset and convert it to conversational format as well.

We have to use Unsloth's `standardize_sharegpt` function to fix up the format of the dataset first.

In [21]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(non_reasoning_dataset)

non_reasoning_conversations = tokenizer.apply_chat_template(
    list(dataset["conversations"]),
    tokenize = False,
)

Unsloth: Standardizing formats (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

Let's see the first row

In [22]:
non_reasoning_conversations[0]

'<|im_start|>user\nExplain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. \n\nFurthermore, add the requirement that the code must be written in a language that does not support short-circuit evaluation natively, forcing the test taker to implement their own logic for short-circuit evaluation.\n\nFinally, delve into the concept of truthiness and falsiness in programming languages, explaining how it affects the evaluation of boolean expressions. Add the constraint that the test taker must write code that handles cases where truthiness and falsiness are implemented differently across different programming languages.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</th

Now let's see how long both datasets are:

In [23]:
print(len(clinical_summaries))
print(len(reasoning_conversations))
print(len(non_reasoning_conversations))

191
19252
100000


The non reasoning dataset is much longer. I need the model to retain some simple chat capabilities, but I specifically want a clinical reasoning model.

Let's define a ratio of chat only data. The goal is to define some mixture of both sets of data.

Let's select 75% reasoning and 25% chat based:

In [24]:
clinical_ratio = 0.50
reasoning_ratio = 0.15
non_reasoning_ratio = 0.35

target_size = len(clinical_summaries) * 2
clinical_n = int(target_size * clinical_ratio)
reasoning_n = int(target_size * reasoning_ratio)
non_reasoning_n = int(target_size * non_reasoning_ratio)

print(clinical_n)
print(reasoning_n)
print(non_reasoning_n)

191
57
133


In [25]:
import pandas as pd
clinical_subset = pd.DataFrame(clinical_summaries).sample(clinical_n, random_state = config.env.seed)
clinical_subset['type'] = 'clinical'

reasoning_subset = pd.DataFrame(reasoning_conversations).sample(reasoning_n, random_state = config.env.seed)
reasoning_subset['type'] = 'reasoning'

non_reasoning_subset = pd.DataFrame(non_reasoning_conversations).sample(non_reasoning_n, random_state = config.env.seed)
non_reasoning_subset['type'] = 'non_reasoning'

print(len(clinical_subset))
print(len(reasoning_subset))
print(len(non_reasoning_subset))

191
57
133


Finally combine the datasets:

In [26]:
data = pd.concat([
    clinical_subset,
    reasoning_subset,
    non_reasoning_subset
])

data.columns = ["text", "type"]
data

,text,type
152,<|im_start|>user\nSummarize the following clin...,clinical
75,<|im_start|>user\nSummarize the following clin...,clinical
158,<|im_start|>user\nSummarize the following clin...,clinical
66,<|im_start|>user\nSummarize the following clin...,clinical
60,<|im_start|>user\nSummarize the following clin...,clinical
...,...,...
14952,<|im_start|>user\nDescribe the method to find ...,non_reasoning
39331,<|im_start|>user\nHow can I rewrite this code ...,non_reasoning
48587,<|im_start|>user\nWhat happens when photons in...,non_reasoning
83594,<|im_start|>user\nExplain the meaning of the e...,non_reasoning


In [27]:
from datasets import Dataset
combined_dataset = Dataset.from_pandas(pd.DataFrame(data))
combined_dataset = combined_dataset.shuffle(seed = config.env.seed)

if False:
    if config.model.training_mode == "smoke_test":
        combined_dataset = combined_dataset.select(range(30))

combined_dataset

Dataset({
    features: ['text', 'type', '__index_level_0__'],
    num_rows: 381
})

In [28]:
# Split into train and test
from datasets import ClassLabel

# Create a ClassLabel feature from existing unique values
labels = combined_dataset.features["type"].names if hasattr(combined_dataset.features["type"], "names") else None

if labels is None:
    labels = list(set(combined_dataset["type"]))

combined_dataset = combined_dataset.cast_column(
    "type",
    ClassLabel(names=labels)
)

combined_dataset = combined_dataset.train_test_split(
    test_size = config.env.test_size,
    seed = config.env.seed,
    stratify_by_column='type'
)

combined_dataset

Casting the dataset:   0%|          | 0/381 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'type', '__index_level_0__'],
        num_rows: 342
    })
    test: Dataset({
        features: ['text', 'type', '__index_level_0__'],
        num_rows: 39
    })
})

In [29]:
combined_dataset['train'].features["type"]

ClassLabel(names=['reasoning', 'non_reasoning', 'clinical'])

In [30]:
display(combined_dataset['train'].to_pandas()['type'].value_counts(normalize=1))
display(combined_dataset['test'].to_pandas()['type'].value_counts(normalize=1))

type
2    0.502924
1    0.347953
0    0.149123
Name: proportion, dtype: float64

type
2    0.487179
1    0.358974
0    0.153846
Name: proportion, dtype: float64

<a name="Train"></a>
## Train the model


In [31]:
batch_size      = config.model.batch_size * config.model.gradient_acc
dataset_size    = combined_dataset['train'].shape[0]
steps_per_epoch = (dataset_size // batch_size)

print(
    f"Effective batch size is: {batch_size} \n"
    f"The dataset has {dataset_size} examples \n"
    f"One full pass (epoch) through the dataset corresponds to roughly\n {dataset_size}/{batch_size} ≈ {steps_per_epoch} optimization steps."
)

Effective batch size is: 4 
The dataset has 342 examples 
One full pass (epoch) through the dataset corresponds to roughly
 342/4 ≈ 85 optimization steps.


In [33]:
# Train mode hyperparameters
if config.model.training_mode == "smoke_test":
    save_strategy = "steps" # or "no"
    eval_strategy = "steps" # or "no"
    max_steps = 10
    save_steps = 2
    eval_steps = 2

elif config.model.training_mode == "steps":
    save_strategy = "steps"
    eval_strategy = "steps"
    epoch_fraction = 0.75
    max_steps = int(steps_per_epoch * epoch_fraction)
    save_steps = int(steps_per_epoch * .33)
    eval_steps = int(steps_per_epoch * .33)

else: # config.model.training_mode == "epochs"
    save_strategy = "epoch"
    eval_strategy = "epoch"
    max_steps = -1  # When max_steps > 0 it overrides the "epoch" strategy
    save_steps = 0  # IGNORED HYPERPARAM
    eval_steps = 0  # IGNORED HYPERPARAM

In [34]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = combined_dataset['train'],
    eval_dataset = combined_dataset['test'],
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = config.model.batch_size,
        per_device_eval_batch_size = config.model.batch_size,
        gradient_accumulation_steps = config.model.gradient_acc, # Use GA to mimic batch size!
        eval_accumulation_steps = config.model.gradient_acc,

        save_strategy = save_strategy,
        eval_strategy = eval_strategy,
        max_steps = max_steps,
        save_steps = save_steps,
        eval_steps = eval_steps,

        num_train_epochs = config.model.num_train_epochs,
        warmup_steps = 5,
        learning_rate = 5e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",

        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        load_best_model_at_end = True,
        save_total_limit = 2, # NOTE: when `load_best_model_at_end == True`, the best performing model is always kept

        seed = config.env.seed,
        report_to = "trackio" if config.env.enable_trackio else "none",
        push_to_hub = config.model.push_to_hub,
        hub_model_id = config.model.fine_tuned_model_repo_id,
        hub_strategy = "checkpoint",
        run_name = f"{config.model.project_name}-{config.model.checkpoint.split('/')[-1]}",
        padding_free  = False, # Set to True if > 17 GB VRAM
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/342 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/39 [00:00<?, ? examples/s]

In [35]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
11.84 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [36]:
# clean gpu and garbage collect
torch.cuda.empty_cache()
import gc
gc.collect()

105722

In [37]:
if not config.model.resume_training:
    print("Training from scratch...")
    trainer_stats = trainer.train()
else:
    print("Resuming training...")
    trainer.train(resume_from_checkpoint=f"{config.model.fine_tuned_model_repo_id}/last-checkpoint")

Training from scratch...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 342 | Num Epochs = 3 | Total steps = 258
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 128,450,560 of 14,896,757,760 (0.86% trained)


* Trackio project initialized: huggingface
* Trackio metrics will be synced to Hugging Face Dataset: Francesco-A/trackio-dataset
* Found existing space: https://huggingface.co/spaces/Francesco-A/trackio
* View dashboard by going to: https://Francesco-A-trackio.hf.space/


* Created new run: sft-tests-Qwen3-14B
Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.601200,0.856208
2,1.134400,0.808859
3,0.960800,0.798296


Exception in thread Thread-25 (_init_client_background):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/trackio/run.py", line 111, in _init_client_background
    self._batch_sender()
  File "/usr/local/lib/python3.12/dist-packages/trackio/run.py", line 82, in _batch_sender
    self._client.predict(
  File "/usr/local/lib/python3.12/dist-packages/gradio_client/client.py", line 499, in predict
    ).result()
      ^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio_client/client.py", line 1484, in result
    return super().result(timeout=timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/concurrent/futures/_base.py", line 456, in result
    return self.__get_result()
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/pytho

* Run finished. Uploading logs to Trackio (please wait...)


In [38]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

5741.1703 seconds used for training.
95.69 minutes used for training.
Peak reserved memory = 14.396 GB.
Peak reserved memory for training = 2.556 GB.
Peak reserved memory % of max memory = 98.853 %.
Peak reserved memory for training % of max memory = 17.551 %.


<a name="Inference"></a>
## Inference
Let's run the model via Unsloth native inference! According to the `Qwen-3` team, the recommended settings for reasoning inference are `temperature = 0.6, top_p = 0.95, top_k = 20`

For normal chat based inference, `temperature = 0.7, top_p = 0.8, top_k = 20`

In [ ]:
messages = [
    {"role" : "user", "content" : "Summarize the following clinical trial report for policy makers:\n\nTITLE: Danish Cardiogenic Shock Trial\nSUMMARY: Cardiogenic shock a serious complication of a heart attack (myocardial infarction). Despite rapid invasive treatment, circulatory support using positive inotropes and mechanical support with intra-aortic balloon counterpulsation (IABP), and evaluation of several new treatments during the last decade, the mortality in patients with cardiogenic shock still exceeds 50%. An alternative to current management is restoration of the volume of blood pumped by the heart (cardiac output) using a ventricular assist device. In the acute setting this is difficult but can be done using the Impella device which is a catheter-based, axial flow pump that pumps blood directly from the left ventricle into the circulation thereby restoring blood flow to the failing organs. In 2012 a more powerful Impella has been introduced that is able to deliver 3.5l/min (approximately 75% of a normal cardiac output). The hypothesis of the current study is to reduce mortality and morbidity of patients with cardiogenic shock using the Impella CP. The study will be carried out as a randomized multicenter study where eligible patients will be randomized to receive conventional circulatory support or support with the Impella device and inotropic support if needed. A total of 360 patients are planned to be enrolled, and the primary endpoint will be death."}
]

# messages = [
#     {"role" : "user", "content" : "Solve (x + 2)^2 = 0."}
# ]

In [41]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 256, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

**Danish Cardiogenic Shock Trial: Summary for Policy Makers**

Cardiogenic shock is a severe complication of heart attacks, with a mortality rate exceeding 50% despite current treatments. This study aims to evaluate the effectiveness of the Impella CP device, a catheter-based pump that restores blood flow to the heart, in reducing mortality and morbidity in patients with cardiogenic shock. The study will be a randomized multicenter trial involving 360 patients, with the primary endpoint being death. The Impella CP device has the potential to provide a more effective treatment option for patients with cardiogenic shock.<|im_end|>


In [42]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = True, # Enable thinking
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1024, # Increase for longer outputs!
    temperature = 0.6, top_p = 0.95, top_k = 20, # For thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<think>
* Key technical facts: Cardiogenic shock is a life-threatening complication of heart attack with a mortality rate exceeding 50%, despite current treatments (inotropes, IABP). The Impella CP is a catheter-based, axial flow pump capable of delivering 3.5L/min (75% of normal cardiac output). A randomized, multicenter trial (n=360) is comparing Impella CP support to conventional support. Primary endpoint is mortality.
* Implications for the audience: High mortality from cardiogenic shock represents a significant public health burden. The Impella CP offers a potentially superior treatment option. Successful implementation of Impella CP could significantly reduce healthcare costs associated with treating cardiogenic shock and improve patient outcomes. Policy support for research and potential future adoption of Impella CP in clinical guidelines could be justified.
* Structural plan: The report will concisely define cardiogenic shock, summarize the problem with current treatments, exp

<a name="Save"></a>
## Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [43]:
if config.model.save_local:
    model.save_pretrained("qwen_lora")  # Local saving
    tokenizer.save_pretrained("qwen_lora")

if config.model.push_to_hub:
    # model.push_to_hub(config.model.fine_tuned_model_repo_id)
    # tokenizer.push_to_hub(config.model.fine_tuned_model_repo_id)
    trainer.push_to_hub("End of training")

0it [00:00, ?it/s]

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False:
    model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False:
    model.save_pretrained_merged("qwen_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: # Pushing to HF Hub
    model.push_to_hub_merged("HF_USERNAME/qwen_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False: # Pushing to HF Hub
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False:
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False:
    model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: # Pushing to HF Hub
    model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).